In [0]:
# products_df
products_df = spark.createDataFrame(
    [
        (1, "Apple Juice", "Beverages"),
        (2, "Orange Juice", "Beverages"),
        (3, "Chocolate Bar", "Snacks"),
        (4, "Potato Chips", "Snacks"),
        (5, "Fresh Strawberries", "Fruits"),
    ],
    ["product_id", "name", "category"],
)


# sales_df
sales_df = spark.createDataFrame(
    [
        (1, 1, 10, 20.0),
        (2, 1, 5, 10.0),
        (3, 2, 8, 16.0),
        (4, 3, 2, 4.0),
        (5, 4, 15, 30.0),
    ],
    ["sale_id", "product_id", "quantity", "revenue"],
)


# inventory_df
inventory_df = spark.createDataFrame(
    [
        (1, 50, "Warehouse1"),
        (2, 40, "Warehouse1"),
        (3, 30, "Warehouse1"),
        (4, 20, "Warehouse1"),
        (5, 10, "Warehouse1"),
    ],
    ["product_id", "stock", "warehouse"],
)
sales_agg = sales_df.groupBy("product_id").agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue")
)

final_df = products_df \
    .join(sales_agg, "product_id", "left") \
    .join(inventory_df, "product_id", "left")


final_df = final_df \
    .withColumn("total_quantity", coalesce(col("total_quantity"), lit(0))) \
    .withColumn("total_revenue", coalesce(col("total_revenue"), lit(0))) \
    .withColumn("total_stock", coalesce(col("stock"), lit(0)))


final_df.select('product_id','name','category', coalesce(col("total_quantity"), lit(0)).alias("total_quantity"),
                coalesce(col("total_revenue"), lit(0)).alias("total_revenue"),
                coalesce(col("stock"), lit(0)).alias("total_stock")).display()